# 🧬 VIDEO-FREE · 6. Entrenar tus personajes (para dejar toda la noche)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/6_entrenar_personaje.ipynb)

Enseña a la IA **cómo son tu cara y la de tu actriz** para que el director (cuaderno 5) los cree en **cualquier pose y lugar**: de espaldas caminando al auto, de lado manejando, de lejos en la carretera. Todo sin perder el parecido.

**Qué hace, solo y en orden:**
1. Prepara los recortes de `medios/personajes/` y tus fotos reales, si las subes a tu Drive en `VIDEO-FREE/entrenamiento/actor/` o `.../actriz/` (10 a 20 fotos con distintas luces y ángulos mejoran mucho el parecido).
2. **Crea fotos nuevas con la IA**: tu cara exacta, en el Camaro, en la carretera, en la ciudad de noche, en el campo, con luces distintas. Así aprende la cara y no el fondo de estudio.
3. **Entrena** (DreamBooth LoRA sobre RealVisXL 5.0) cada personaje. Guarda un punto de control en tu Drive cada pocos cientos de pasos: si Colab se desconecta, vuelve a ejecutar y sigue donde iba.
4. Prueba los últimos puntos de control lado a lado, para elegir el que más se parece.
5. Guarda el personaje entrenado para el cuaderno 5.

**Para dejarlo de noche:** elige `PERSONAJES = ambos`, luego *Entorno de ejecución → Ejecutar todo*, y deja la pestaña abierta con el computador despierto.
- **Tiempo aproximado en T4 gratis:** 15–20 min de fotos y 1–1,5 h de entrenamiento por personaje con 2400 pasos.
- **Entrenar más mejora el parecido, no la velocidad.** La velocidad del video la dan los modelos rápidos del cuaderno 5 (FastWan y RealVisXL Lightning). Demasiados pasos pueden «sobreentrenar» (caras rígidas, todas iguales); para eso está la comparación del paso 4.
- **Licencias:** RealVisXL (OpenRAIL++), script de Diffusers (Apache 2.0), PEFT (Apache 2.0), bitsandbytes (MIT).
- Entrena **solo con tu cara o con la de alguien que te dé permiso**. Nada de caras de famosos.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
#@markdown Guarda copias en tu Google Drive (Colab te pedirá permiso para acceder a Drive).
USAR_DRIVE = True  #@param {type:"boolean"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=USAR_DRIVE)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar (2–3 min)
!pip install -q -U "diffusers>=0.35" transformers accelerate peft bitsandbytes datasets
import diffusers, urllib.request, importlib, director
importlib.reload(director)
version = diffusers.__version__.split(".dev")[0]
urllib.request.urlretrieve(f"https://raw.githubusercontent.com/huggingface/diffusers/v{version}/examples/dreambooth/train_dreambooth_lora_sdxl.py",
                           "/content/entrenar_lora_sdxl.py")
print("Diffusers", version, "· script de entrenamiento listo")

In [ ]:
#@title 3 · Qué entrenar
PERSONAJES = "ambos"  #@param ["ambos", "actor", "actriz"]
FOTOS_IA = 40  #@param {type:"slider", min:0, max:80, step:10}
#@markdown Fotos nuevas que la IA crea con la cara de cada personaje en otros lugares y luces.
PASOS = 2400  #@param {type:"slider", min:400, max:4000, step:200}
GUARDAR_CADA = 400  #@param {type:"slider", min:200, max:1000, step:100}
RANGO = 16  #@param [8, 16, 32] {type:"raw"}
#@markdown Más rango = aprende más detalle (y el archivo pesa más).
APRENDIZAJE = 1e-4  #@param {type:"number"}
import glob, json, shutil
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display
LISTA = ["actor", "actriz"] if PERSONAJES == "ambos" else [PERSONAJES]
PALABRAS = {"actor": "jsnv man", "actriz": "mrla woman"}  # palabra clave de cada personaje en los prompts
QUIEN = {"actor": "young man", "actriz": "young woman"}
RES = 1024 if VRAM >= 22 else 768
DATOS = {q: Path(f"/content/datos_{q}") for q in LISTA}
print("Personajes:", ", ".join(f"{q} ({PALABRAS[q]})" for q in LISTA), f"· {RES} px · {PASOS} pasos cada uno")

In [ ]:
#@title 4 · Preparar los recortes y tus fotos
FONDOS = [(126, 126, 126), (212, 188, 158), (58, 68, 88), (232, 232, 226), (148, 108, 78)]

def sobre_fondo(rgba, color):
    fondo = Image.new("RGB", rgba.size, color)
    sombra = Image.new("RGB", rgba.size, tuple(max(0, c - 55) for c in color))
    fondo = Image.composite(sombra, fondo, Image.linear_gradient("L").resize(rgba.size).point(lambda v: int(v * 0.4)))
    fondo.paste(rgba, (0, 0), rgba)
    return fondo

def a_cuadrado(img, color):
    s = max(img.size)
    lienzo = Image.new("RGB", (s, s), color)
    lienzo.paste(img, ((s - img.width) // 2, (s - img.height) // 2))
    return lienzo.resize((RES, RES), Image.LANCZOS)

def guardar_foto(carpeta, img, espejo=True):
    for invertir in ((False, True) if espejo else (False,)):
        n = len(list(carpeta.glob("*.png")))
        (ImageOps.mirror(img) if invertir else img).save(carpeta / f"{n:03d}.png")

def recortes_de(quien):
    return sorted(glob.glob(f"/content/VIDEO-FREE/medios/personajes/{quien}_*.png"))

for quien in LISTA:
    carpeta = DATOS[quien]
    shutil.rmtree(carpeta, ignore_errors=True); carpeta.mkdir(parents=True)
    for k, ruta in enumerate(recortes_de(quien)):
        rgba = Image.open(ruta).convert("RGBA")
        rgba = rgba.crop(rgba.getchannel("A").point(lambda v: 255 if v > 8 else 0).getbbox())
        for j in range(2):
            color = FONDOS[(k + j * 2) % len(FONDOS)]
            guardar_foto(carpeta, a_cuadrado(sobre_fondo(rgba, color), color))
            if rgba.height > rgba.width * 1.3:  # foto de cuerpo entero: también un primer plano de la cara
                cara = rgba.crop((0, 0, rgba.width, min(rgba.height, rgba.width)))
                guardar_foto(carpeta, a_cuadrado(sobre_fondo(cara, color), color))
    propias = []
    mias = comun.en_drive(f"entrenamiento/{quien}")
    if mias:
        propias = [f for f in sorted(mias.iterdir()) if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]
    for ruta in propias:
        img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
        guardar_foto(carpeta, ImageOps.fit(img, (RES, RES), Image.LANCZOS, centering=(0.5, 0.35)))
    print(f"{quien}: {len(recortes_de(quien))} recortes + {len(propias)} fotos tuyas → {len(list(carpeta.glob('*.png')))} imágenes")
    if len(propias) < 8:
        print(f"   Consejo: con 10–20 fotos reales en Drive (VIDEO-FREE/entrenamiento/{quien}/) el parecido mejora mucho.")

In [ ]:
#@title 5 · Crear fotos de entrenamiento con la IA (tu cara exacta en otros lugares y luces)
import gc, torch
from diffusers import StableDiffusionXLInpaintPipeline, DPMSolverMultistepScheduler
pipe = None
for quien in LISTA:
    if not FOTOS_IA:
        break
    recortes = [Image.open(r).convert("RGBA") for r in recortes_de(quien)]
    hechas = 0
    for k, v in enumerate(director.variaciones_entrenamiento(FOTOS_IA, semilla=3 if quien == "actor" else 5)):
        nombre_f = f"entrenamiento/{quien}_ia/{k:03d}.png"
        previa = comun.en_drive(nombre_f)
        if previa:
            img = Image.open(previa).convert("RGB")
        else:
            if pipe is None:
                pipe = StableDiffusionXLInpaintPipeline.from_pretrained("OzzyGT/RealVisXL_V4.0_inpainting", torch_dtype=torch.float16,
                                                                        variant="fp16").to("cuda")
                pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True,
                                                                         algorithm_type="sde-dpmsolver++")
            recorte = recortes[k % len(recortes)]
            caja = recorte.getchannel("A").point(lambda x: 255 if x > 8 else 0).getbbox()
            cuerpo_entero = caja and (caja[3] - caja[1]) > (caja[2] - caja[0]) * 1.3
            encuadre = v["encuadre"]
            if encuadre.startswith("cuerpo") and not cuerpo_entero:  # un recorte de cara no sirve para cuerpo entero
                encuadre = "medio_izquierda" if encuadre.endswith("izquierda") else "medio_derecha"
            lienzo, mascara = director.componer_recorte(recorte, 1024, 1024, encuadre)
            prompt = f"photo of a {QUIEN[quien]} {v['lugar']}, {v['luz']}, {director.ESTILO_FOTO}"
            pintado = pipe(prompt=prompt, negative_prompt=director.NEGATIVO_FOTO, image=lienzo, mask_image=mascara, width=1024, height=1024,
                           strength=0.99, num_inference_steps=25, guidance_scale=6.0,
                           generator=torch.Generator("cuda").manual_seed(v["semilla"])).images[0].resize((1024, 1024))
            img = Image.composite(pintado, lienzo, mascara)
            local = comun.TRABAJO / f"{quien}_ia_{k:03d}.png"
            img.save(local); comun.a_drive(local, nombre_f)
        guardar_foto(DATOS[quien], img.resize((RES, RES), Image.LANCZOS), espejo=False)
        hechas += 1
    print(f"{quien}: {hechas} fotos creadas con la IA (guardadas en Drive: VIDEO-FREE/entrenamiento/{quien}_ia/)")
    muestra = sorted(DATOS[quien].glob("*.png"))[-min(12, hechas):]
    if muestra:
        display(Image.open(director.hoja_de_contactos(muestra, f"/content/muestra_{quien}.jpg", columnas=6, ancho=180)))
if pipe is not None:
    del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
#@title 6 · Entrenar (retoma solo si Colab se desconectó)
SALIDAS = {}
for quien in LISTA:
    salida = comun.carpeta_drive(f"lora/entrenando_{quien}") or Path(f"/content/lora_{quien}")
    salida.mkdir(parents=True, exist_ok=True)
    SALIDAS[quien] = salida
    if (salida / "pytorch_lora_weights.safetensors").exists():
        print(f"{quien}: ya estaba entrenado en {salida}. Bórralo de Drive si quieres entrenar de nuevo."); continue
    print(f"Entrenando {quien} ({PALABRAS[quien]}) con {len(list(DATOS[quien].glob('*.png')))} imágenes…")
    comun.sh(f'python /content/entrenar_lora_sdxl.py --pretrained_model_name_or_path SG161222/RealVisXL_V5.0 --variant fp16 '
             f'--instance_data_dir "{DATOS[quien]}" --instance_prompt "a photo of {PALABRAS[quien]}" --output_dir "{salida}" '
             f'--resolution {RES} --train_batch_size 1 --gradient_accumulation_steps 1 --gradient_checkpointing --use_8bit_adam '
             f'--mixed_precision fp16 --learning_rate {APRENDIZAJE} --lr_scheduler constant --lr_warmup_steps 0 '
             f'--max_train_steps {PASOS} --rank {RANGO} --seed 0 --checkpointing_steps {GUARDAR_CADA} --checkpoints_total_limit 3 '
             f'--resume_from_checkpoint latest --report_to tensorboard')
    print(f"{quien}: entrenamiento terminado")

In [ ]:
#@title 7 · Comparar los últimos puntos de control
import gc, torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V5.0_Lightning", torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")
PRUEBAS = ["driving a black 1969 convertible on a desert highway at sunset, wind in the hair",
           "close-up portrait singing with emotion at night, dashboard light on the face",
           "walking toward a parked black 1969 convertible, seen from the side, golden hour"]
CANDIDATOS = {}
for quien in LISTA:
    salida = SALIDAS[quien]
    candidatos = [(c.name.split("-")[-1], c) for c in sorted(salida.glob("checkpoint-*"), key=lambda c: int(c.name.split("-")[-1]))][-2:]
    if (salida / "pytorch_lora_weights.safetensors").exists():
        candidatos.append(("final", salida))
    CANDIDATOS[quien] = dict(candidatos)
    fotos = []
    for etiqueta, carpeta in candidatos:
        pipe.unload_lora_weights()
        pipe.load_lora_weights(str(carpeta), weight_name="pytorch_lora_weights.safetensors")
        for k, pr in enumerate(PRUEBAS):
            img = pipe(prompt=f"photo of {PALABRAS[quien]} {pr}, {director.ESTILO_FOTO}", negative_prompt=director.NEGATIVO_FOTO,
                       width=1024, height=1024, num_inference_steps=6, guidance_scale=1.5,
                       generator=torch.Generator("cuda").manual_seed(k)).images[0]
            f = comun.TRABAJO / f"prueba_{quien}_{etiqueta}_{k}.jpg"; img.save(f); fotos.append(f)
    print(f"{quien} · filas (de arriba abajo):", ", ".join(e for e, _ in candidatos))
    display(Image.open(director.hoja_de_contactos(fotos, comun.TRABAJO / f"pruebas_{quien}.jpg", columnas=len(PRUEBAS), ancho=300)))
del pipe; gc.collect(); torch.cuda.empty_cache()
print("Elige la fila que más se parece y escríbela en la celda 8 («final» o el número de pasos).")

In [ ]:
#@title 8 · Guardar los personajes entrenados
USAR_ACTOR = "final"  #@param {type:"string"}
USAR_ACTRIZ = "final"  #@param {type:"string"}
for quien in LISTA:
    eleccion = (USAR_ACTOR if quien == "actor" else USAR_ACTRIZ).strip() or "final"
    carpeta = CANDIDATOS[quien].get(eleccion)
    if carpeta is None:
        print(f"{quien}: no encuentro «{eleccion}». Opciones: {', '.join(CANDIDATOS[quien])}"); continue
    archivo = carpeta / "pytorch_lora_weights.safetensors"
    comun.a_drive(archivo, f"lora/{quien}.safetensors")
    mb = archivo.stat().st_size / 2**20
    if mb < 95:
        comun.guardar(f"medios/lora/{quien}.safetensors", archivo.read_bytes(), f"Personaje entrenado: {quien}")
    else:
        print(f"{quien}: pesa {mb:.0f} MB, queda solo en tu Drive (GitHub no acepta más de 100 MB).")
    info = {"palabra": PALABRAS[quien], "base": "SG161222/RealVisXL_V5.0", "pasos": eleccion if eleccion != "final" else PASOS,
            "rango": RANGO, "resolucion": RES, "imagenes": len(list(DATOS[quien].glob("*.png")))}
    comun.guardar(f"medios/lora/{quien}.json", json.dumps(info, ensure_ascii=False, indent=1), f"Personaje entrenado: {quien} (datos)")
    print(f"{quien}: guardado ({eleccion})")
print("Listo. En el cuaderno 5 elige MODO_PERSONAJE = «entrenado (pose libre)».")